In [ ]:
from bctools.io import InstrumentResponse
from bctools.loc import LocalLocTable
from bctools.spectra.spectrum import BandFunction,Comptonized
import os
import math
import multiprocessing
import itertools
import pickle 
import time
import multiprocessing
import numpy as np
from bctools.loc import TSMap, NormLocLike
import astropy.units as u
from astropy.coordinates import SkyCoord
from scipy.stats import chi2
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt



run_name = "run10"
irf_path = "/data/models/irf_summed_"+run_name+".h5"
dir_path = "/data/test_newrepo/"

In [ ]:
# Convolve an full instrument response with a hypothetical spectrum
#irf_path = "/data/models/irf_summed.h5"
#irf_path = "/data/models/run_1_noeffect_irf_summed.h5"

# The code is inspired by the bc-tools tutorial.

load_from_file = True

if load_from_file:

    with open('/data/models/LUTS/soft_lut_' + run_name + '.pkl', 'rb') as f:
        soft_sky_loctable = pickle.load(f)
    
    with open('/data/models/LUTS/medium_lut_' + run_name + '.pkl', 'rb') as f:
        medium_sky_loctable = pickle.load(f)
    
    with open('/data/models/LUTS/hard_lut_' + run_name + '.pkl', 'rb') as f:
        hard_sky_loctable = pickle.load(f)
    
else:

    with InstrumentResponse(irf_path) as irf: 
        
        # Hypothetical spectrum
        # This normalization corresponds to 1 ph/cm2/s between 50-300 keV
        #spectrum = PowerLaw(60,2)
        #spectrum = PowerLaw._from_megalib(['PowerLaw',10,10000,2],"5.0")
        soft_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1.9,-3.7,230],"10.0")
        medium_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1,-2.3,699.9],"10.0")
        hard_spectrum = Comptonized._from_megalib(['Comptonized',10,10000,-0.5,1500],"10.0")
        
        # In this case we integrate the rate from all energy channels. 
        # You can subdivide the data into multiple energy channel groups
        soft_local_loctable = LocalLocTable.from_irf(irf, soft_spectrum,energy_channels = 1) # [80,2000]
        medium_local_loctable = LocalLocTable.from_irf(irf, medium_spectrum,energy_channels = 1)
        hard_local_loctable = LocalLocTable.from_irf(irf, hard_spectrum,energy_channels = 1)
        
    # The local_loctable contains the expected rates in spacecraft coordinates
    # We now need to use this to estimate the total expected counts in sky coordinate for
    # the full duration of an event. 
    # In this case we simply have a 1 second event and specifying the attitude by a quaternion
    # ([0,0,0,1] corresponds to the identity rotation). You can have multiple attitude-duration
    # pairs to correctly model long duration events.
    soft_sky_loctable = soft_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    medium_sky_loctable = medium_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    hard_sky_loctable = hard_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    
    #Store LUT
    
    import pickle
    
    # Salvataggio su file
    with open('/data/models/LUTS/soft_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(soft_sky_loctable, f)
    with open('/data/models/LUTS/medium_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(medium_sky_loctable, f)
    with open('/data/models/LUTS/hard_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(hard_sky_loctable, f)

In [ ]:
#soft_local_loctable.channel_mask("BGO_Y0")

In [ ]:

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

In [ ]:
print(f"Soft Look-up tables: {soft_sky_loctable.labels}")
print(f"Medium Look-up tables: {medium_sky_loctable.labels}")
print(f"Hard Look-up tables: {hard_sky_loctable.labels}")



In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').plot();

In [ ]:
import pickle
if True:

    # Salva l'array in un file usando pickle
    with open("/data/bctools_exp_medium_Y1.pkl", "wb") as f:
        pickle.dump(medium_sky_loctable.get_expectation_map('BGO_Y1').data, f)


In [ ]:
#test with LUTs
import pickle 

file_name_test = "run57_mix_mega_shared"

file_path = dir_path+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)


In [ ]:
loaded_array_test[2]

In [ ]:
import numpy as np

filter_flux = 1
filter_spectra = 1

analysis_spectrum = "medium"

# These values lead to a mean sigma between 10-15 and were used for the paper
if analysis_spectrum == "soft":
    min_flux = 14
    max_flux = 16
    spectrum_string = "230"
    
elif analysis_spectrum == "medium" :
    min_flux = 1
    max_flux = 3
    spectrum_string = "699"
    
elif analysis_spectrum == "hard":
    min_flux = 1
    max_flux = 1.6
    spectrum_string = "1500"

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] > min_flux and grb['flux'] <= max_flux:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        #if  grb['spectra'] == 'medium':
        if  spectrum_string in grb['spectrum']:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape

In [ ]:
test_number = 10000
random_indices = np.random.permutation(len(loaded_array_test))

# Use the random index array to shuffle the original array
shuffled_loaded_array = loaded_array_test[random_indices]

if(len(loaded_array_test)<=test_number):
    train_dataset = []

    test_dataset = shuffled_loaded_array
else:
    train_dataset = shuffled_loaded_array[:len(shuffled_loaded_array)-test_number]

    test_dataset = shuffled_loaded_array[len(shuffled_loaded_array)-test_number:]

                                       

In [ ]:
import numpy as np

def process_source(
    grb,
    band="auto",                  # "soft" | "medium" | "hard" | "auto" | or a list/tuple, e.g. ("soft","hard")
):
    """
    Parameters
    ----------
    grb : dict
        Must contain: 
          - 'coord' = (theta, phi)
          - 'counts' = array/list with at least 6 channels (will be reordered below)
          - optional 'spectrum'
    band : str | Iterable[str]
        "soft", "medium", "hard", "auto" (default), or a collection of bands to try.
        With "auto" it will compare all three and pick the one with max TS.
   
    Returns
    -------
    list with:
      [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist,
       max_sqrt_ts, cont_radius, best_ts, original_ts_value, cont_area, b_counts_first, chosen_band]
    """

    # --- True coordinates from GRB metadata
    theta_real = float(grb['coord'][0])
    phi_real   = float(grb['coord'][1])

    # --- Reorder input counts as in original code
    s_counts = np.array([grb['counts'][3], grb['counts'][2], grb['counts'][5],
                         grb['counts'][4], grb['counts'][1], grb['counts'][0]], dtype=float)

    # --- Simulated background counts (different IRF order)
    b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617], dtype=float)
    b_counts = np.array([b_sim[3], b_sim[2], b_sim[5], b_sim[4], b_sim[1], b_sim[0]], dtype=float) * 20

    # --- Optionally add Poisson fluctuations to source counts
   
    random_bkg = np.random.poisson(lam=b_counts)
    s_counts = s_counts + random_bkg

    # --- Decide which bands to analyze
    if isinstance(band, str):
        band = band.lower()
        if band == "auto":
            bands_to_try = ("soft", "medium", "hard")
        elif band in ("soft", "medium", "hard"):
            bands_to_try = (band,)
        else:
            raise ValueError(f'band="{band}" not valid: use "soft", "medium", "hard" or "auto".')
    else:
        # Accept tuple/list of bands
        bands_to_try = tuple(b.lower() for b in band)
        for b in bands_to_try:
            if b not in ("soft", "medium", "hard"):
                raise ValueError(f'Invalid band in list: "{b}". Allowed: soft/medium/hard.')

    # --- Map band names to localization tables (assume they exist in global scope)
    band_to_table = {
        "soft":   soft_sky_loctable,
        "medium": medium_sky_loctable,
        "hard":   hard_sky_loctable,
    }

    # --- Run localization for each selected band
    results_per_band = {}
    for b in bands_to_try:
        sqrt_ts, ra_loc, dec_loc, cont_radius, ts, loc_tsvalue, cont_area = localize_grb(
            band_to_table[b],
            s_counts,
            b_counts,
            theta_real,
            phi_real
        )
        results_per_band[b] = {
            "sqrt_ts": sqrt_ts,
            "ra_loc": ra_loc,
            "dec_loc": dec_loc,
            "cont_radius": cont_radius,
            "ts": ts,
            "loc_tsvalue": loc_tsvalue,
            "cont_area": cont_area,
        }

    # --- Choose the best band by maximum sqrt_ts
    best_band = max(results_per_band, key=lambda k: results_per_band[k]["sqrt_ts"])
    best = results_per_band[best_band]

    # --- Convert RA/DEC of best localization into spherical coordinates
    theta_loc, phi_loc = ra_dec_to_theta_phi(best["ra_loc"], best["dec_loc"])

    # --- Angular metrics: distance and coordinate offsets
    dist = angular_distance(theta_loc, phi_loc, theta_real, phi_real)
    theta_dist = float(np.abs(theta_loc - theta_real))
    phi_dist   = float(diff_phi(phi_loc, phi_real))

    # --- Package results (note: fixed typo b_s_counts -> b_counts[0])
    result = [
        theta_real,                    # 0 true theta
        phi_real,                      # 1 true phi
        theta_loc,                     # 2 localized theta
        phi_loc,                       # 3 localized phi
        dist,                          # 4 angular distance
        theta_dist,                    # 5 theta difference
        phi_dist,                      # 6 phi difference
        best["sqrt_ts"],               # 7 max sqrt TS
        best["cont_radius"],           # 8 localization contour radius
        best["ts"],                    # 9 TS value
        best["loc_tsvalue"],           # 10 TS value at localization
        best["cont_area"],             # 11 contour area
        float(b_counts[0]),            # 12 first background bin
        best_band,                     # 13 chosen band
    ]

    return result

def localize_grb(sky_loctable,s,b,theta_real,phi_real):
    
    sky_loctable.set_background(b)
    sky_loctable.set_data(s)

    # Define a map of nside = 32. Note that this is a finer resolution
    #that the underlying look-up table, which will be interpolated
    ts = TSMap(nside = 32, coordsys = 'icrs')

    # NormLocLike is a subclass of LocLike and computes
    # a Poisson likelihood for counting instruments. The
    # overall normalization is the only free parameter
    norm_likelihood = NormLocLike(sky_loctable)
   
    # Compute the TS map from one or more LocLikelihood
    ts.compute(norm_likelihood)
    #print("#"+str(theta_real)+"#")
    
    # Correggi theta_real prima di convertirlo
    if theta_real < 0:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 0
    elif theta_real > 180:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 180
    
    theta_rad = np.deg2rad(theta_real)
    phi_rad = np.deg2rad(phi_real)
    
    if theta_rad < 0 or theta_rad > np.pi:
        print(f"theta_rad fuori range: {theta_rad}")


    #print(dir(ts))
    ipix = ts.ang2pix(theta_rad, phi_rad)

    # Get the value at the given coordinates
    #ipix = ts.ang2pix(ts.nside, theta_real * u.deg.to(u.rad), phi_real * u.deg.to(u.rad))
    original_ts_value = ts._data[ipix]

    # Inizializza l'array dei livelli di confidenza
    confidence_levels = np.arange(0, 1.01, 0.01)
    
    # Calcola i raggi di contenimento per ciascun valore di cont
    containment_radius = np.array([
        np.sqrt(ts.error_area(cont=cont) / np.pi).to(u.deg).value
        for cont in confidence_levels
    ])
    
    #original_ts_value = -1
    cont_area = ts.error_area(cont = .9).to(u.deg**2).value
    
    return np.max(ts),ts.best_loc().ra.deg,ts.best_loc().dec.deg,containment_radius,ts,original_ts_value,cont_area

def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi

def diff_phi(a1, a2):
    # Calcola la differenza diretta
    diff = abs(a1 - a2)
    
    # Trova il percorso più breve tenendo conto del ciclo degli angoli
    if diff > 180:
        diff = 360 - diff
    
    return diff

In [ ]:

results_bkg = []
spectra_fitted = 0

def init_worker():
    seed = int.from_bytes(os.urandom(4), "little")
    np.random.seed(seed)

def process_in_parallel(test_dataset,band):
    with multiprocessing.Pool(processes=200, initializer=init_worker) as pool:
        results = pool.starmap(process_source, zip(test_dataset, itertools.repeat(band)))
    return results


results_bkg = process_in_parallel(test_dataset,analysis_spectrum)


In [ ]:
distances = []
theta_distances = []
phi_distances = []
cont_radius_list = []
cont_area_list = []
for res in results_bkg:
    distances.append(res[4])
    theta_distances.append(res[5])
    phi_distances.append(res[6])
    cont_radius_list.append(res[8])
    cont_area_list.append(res[11])




In [ ]:

plt.hist(distances)

In [ ]:
np.mean(distances)

In [ ]:
test_labels = []
for grb in test_dataset:
    theta_real = float(grb['coord'][0])
    phi_real = float(grb['coord'][1])
    test_labels.append([theta_real,phi_real])

test_labels = np.array(test_labels)

In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))

# Frequentist Analysis

In [ ]:
ts_array = []
for r in results_bkg:
    ts_localied = r[7] 
    ts_array.append(ts_localied)
ts_array = np.array(ts_array)
print("Mean TS="+str(np.sqrt(ts_array).mean()))

In [ ]:
def spherical_to_radec_deg(theta_deg, phi_deg):
    """
    Convert spherical coordinates (theta, phi) in degrees to RA and Dec.

    Parameters:
    - theta_deg: polar angle in degrees (0° at north pole)
    - phi_deg: azimuthal angle in degrees (0° at x-axis)

    Returns:
    - ra: Right Ascension in degrees (0 to 360)
    - dec: Declination in degrees (-90 to +90)
    """
    dec = 90.0 - theta_deg
    ra = phi_deg % 360.0
    return ra, dec


In [ ]:
#%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

results = results_bkg[0]
#    result = [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist, max_sqrt, cont_radius,best_ts,loc_ts_value]
theta_real = results[0]
phi_real = results[1]
theta_loc = results[2]
phi_loc = results[3]
ts_max = results[7]
ts_loc = results[9]
print(ts_loc)

print(f"Maximum sqrt(TS) = {np.sqrt(ts_max):.2f}")
print(f"Best estimate: RA = {ts_loc.best_loc().ra:.2f} Dec= {ts_loc.best_loc().dec:.2f}")
print(f"Error area (90% cont.): {ts_loc.error_area(cont = .9).to(u.deg**2):.2f}")
print(f"Equivalent error radius (90% cont.): {np.sqrt(ts_loc.error_area(cont = .9)/np.pi).to(u.deg):.2f}")

img,ax = ts_loc.plot(cont=0.9)
ax.grid(alpha = .5)

real_ra,real_dec = spherical_to_radec_deg(theta_real,phi_real)

# Actual location of simulated source
ax.scatter(real_ra,real_dec,
           color = 'red', transform = ax.get_transform('world'), s = 1);

loc_ra,loc_dec = spherical_to_radec_deg(theta_loc,phi_loc)

# Actual location of simulated source
ax.scatter(loc_ra,loc_dec,
           color = 'magenta', transform = ax.get_transform('world'), s = 1);

In [ ]:

values = np.arange(0, 1.01, 0.01)

fraction_array = []

for value in values:
    count = 0 
    for r in results_bkg:
        max_chi2 = r[7]
        original_ts = r[10]
        if original_ts>max_chi2-chi2.ppf(value, 2).flatten():
            count = count + 1
    fraction_array.append(count/len(results_bkg))
 
 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot della curva empirica
plt.plot(values, fraction_array, label='Model Calibration Curve')

# Aggiunta della unitary line (y = x)
plt.plot(values, values, linestyle='--', color='gray', label='Perfect Calibration (y=x)')

# Etichette e legenda
plt.xlabel('Confidence Level')
plt.ylabel('Fraction < Confidence Level')
plt.title('Calibration Plot')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
fraction_array[90]

In [ ]:
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt

# Confidence level threshold (fixed)
delta_chi2_90 = chi2.ppf(0.9, df=2)

# Define flux bins (adjust range and step as needed)
flux_bins = np.arange(0, 30 + 1, 1)
flux_centers = (flux_bins[:-1] + flux_bins[1:]) / 2
coverage_array = []

# For each flux bin
for i in range(len(flux_bins) - 1):
    count_in = 0
    count_total = 0
    count = 0
    for r in results_bkg:
        flux = float(test_dataset[count]['flux'])
        if flux_bins[i] <= flux < flux_bins[i+1]:
            max_chi2 = r[7]
            original_ts = r[10]
            if original_ts>max_chi2-delta_chi2_90:
                count_in += 1
            count_total += 1
        count = count+1
        
    if count_total > 0:
        coverage = count_in / count_total
    else:
        coverage =  0
    coverage_array.append(coverage)

# Plotting
plt.plot(flux_centers, coverage_array, marker='o')
plt.axhline(0.9, color='gray', linestyle='--', label='Target 90%')
plt.xlabel('Source Flux')
plt.ylabel('Empirical 90% Coverage')
plt.title('Coverage vs. Source Flux at Fixed 90% CL')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
    

for i in range (0,0):#%matplotlib widget
 
    # result = [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist, max_sqrt, cont_radius,best_ts,loc_ts_value]
    theta_real = results_bkg[i][0]
    phi_real = results_bkg[i][1]
    theta_loc = results_bkg[i][2]
    phi_loc = results_bkg[i][3]
    ts_max = results_bkg[i][7]
    ts_loc = results_bkg[i][9]
    print(ts_loc)
    
    print(f"Maximum sqrt(TS) = {np.sqrt(ts_max):.2f}")
    print(f"Best estimate: RA = {ts_loc.best_loc().ra:.2f} Dec= {ts_loc.best_loc().dec:.2f}")
    print(f"Error area (90% cont.): {ts_loc.error_area(cont = .9).to(u.deg**2):.2f}")
    print(f"Equivalent error radius (90% cont.): {np.sqrt(ts_loc.error_area(cont = .9)/np.pi).to(u.deg):.2f}")
    
    img,ax = ts_loc.plot(cont=0.9)
    ax.grid(alpha = .5)
    
    real_ra,real_dec = spherical_to_radec_deg(theta_real,phi_real)
    
    # Actual location of simulated source
    ax.scatter(real_ra,real_dec,
               color = 'red', transform = ax.get_transform('world'), s = 1);
    
    loc_ra,loc_dec = spherical_to_radec_deg(theta_loc,phi_loc)
    
    # Actual location of simulated source
    ax.scatter(loc_ra,loc_dec,
               color = 'magenta', transform = ax.get_transform('world'), s = 1);